# Filmoskop RAG assistant

Minimal project notebook. Current implementation covers plan step 3: two chunking strategies, embeddings, FAISS GPU search, retrieval metrics, and chunking strategy selection.

## Constants and imports

In [ ]:
import json
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_PATH = ROOT / "filmoskop_kb.json"
EMBED_MODEL = "intfloat/multilingual-e5-base"
RECALL_KS = [1, 3, 5, 10, 20]
TOP_K = 20

## Step 3 dependency check

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss

if not hasattr(faiss, "StandardGpuResources") or faiss.get_num_gpus() <= 0:
    raise RuntimeError("FAISS GPU with CUDA is required: StandardGpuResources and get_num_gpus() > 0")

print("FAISS version:", getattr(faiss, "__version__", "unknown"))
print("FAISS GPUs:", faiss.get_num_gpus())

## Minimal corpus loading and cleaning for step 3

This block prepares the input for chunking comparison. Full audit and cleaning report will be expanded in the corresponding notebook sections.

In [ ]:
with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

docs = data["docs"]
queries = data["queries"]

print("Docs:", len(docs), "| Queries:", len(queries))
print("Formats:", dict(Counter(d["format"] for d in docs)))
print("Categories:", dict(Counter(d["category"] for d in docs)))
print("Query kinds:", dict(Counter(q["kind"] for q in queries)))

In [ ]:
def clean_text(raw: str) -> str:
    """Minimally removes markup and normalizes whitespace."""
    text = raw or ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"[`*_#>\-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_docs = []
for d in docs:
    clean_docs.append({
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "text": clean_text(d["raw"]),
    })

pd.DataFrame({
    "metric": ["docs", "queries", "median_clean_len"],
    "value": [len(clean_docs), len(queries), int(np.median([len(d["text"]) for d in clean_docs]))],
})

## Step 3. Two chunking strategies

In [ ]:
CHUNK_STRATEGIES = {
    "small_500_100": {"chunk_size": 500, "chunk_overlap": 100},
    "medium_900_150": {"chunk_size": 900, "chunk_overlap": 150},
}

def make_chunks(clean_docs, chunk_size: int, chunk_overlap: int):
    """Creates chunks with doc_id, title, category, and chunk_id metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in clean_docs:
        parts = splitter.split_text(f"{doc['title']}\n{doc['text']}")
        for i, part in enumerate(parts):
            chunks.append({
                "page_content": part,
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "category": doc["category"],
                "chunk_id": i,
            })
    return chunks

chunk_sets = {
    name: make_chunks(clean_docs, **params)
    for name, params in CHUNK_STRATEGIES.items()
}

pd.DataFrame([
    {
        "strategy": name,
        "chunks": len(chunks),
        "median_len": int(np.median([len(c["page_content"]) for c in chunks])),
        "max_len": max(len(c["page_content"]) for c in chunks),
    }
    for name, chunks in chunk_sets.items()
])

## Step 3. Embeddings and FAISS GPU index

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL, device="cuda")

def encode_passages(texts):
    """Encodes E5 passages with the required passage prefix."""
    vecs = embedder.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True,
    )
    return np.asarray(vecs, dtype="float32")

def encode_queries(texts):
    """Encodes E5 queries with the required query prefix."""
    vecs = embedder.encode(
        ["query: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    return np.asarray(vecs, dtype="float32")

def build_gpu_index(vectors):
    """Builds FAISS GPU IndexFlatIP for normalized embeddings."""
    cpu_index = faiss.IndexFlatIP(vectors.shape[1])
    res = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    gpu_index.add(vectors)
    return gpu_index

indexes = {}
embeddings = {}
for name, chunks in chunk_sets.items():
    embeddings[name] = encode_passages([c["page_content"] for c in chunks])
    indexes[name] = build_gpu_index(embeddings[name])
    print(name, embeddings[name].shape, "index size", indexes[name].ntotal)

## Step 3. Retrieval metrics

Gap queries are excluded from retrieval metrics. Metrics are calculated by unique doc_id values restored from top-K chunks.

In [ ]:
eval_queries = [q for q in queries if q["kind"] != "gap"]
query_vecs = encode_queries([q["query"] for q in eval_queries])

def retrieved_doc_ids(strategy: str, q_vec, k: int):
    """Returns unique doc_id values from top-k retrieved chunks."""
    _, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    doc_ids = []
    seen = set()
    for i in idx[0]:
        doc_id = chunk_sets[strategy][int(i)]["doc_id"]
        if doc_id not in seen:
            doc_ids.append(doc_id)
            seen.add(doc_id)
    return doc_ids

def precision_at_k(found, relevant):
    """Calculates precision for retrieved doc_id values."""
    if not found:
        return 0.0
    rel = set(relevant)
    return sum(doc_id in rel for doc_id in found) / len(found)

def recall_at_k(found, relevant):
    """Calculates recall for retrieved doc_id values."""
    rel = set(relevant)
    if not rel:
        return 0.0
    return len(set(found) & rel) / len(rel)

def mrr(found, relevant):
    """Calculates reciprocal rank of the first relevant doc_id."""
    rel = set(relevant)
    for rank, doc_id in enumerate(found, start=1):
        if doc_id in rel:
            return 1 / rank
    return 0.0

def evaluate_strategy(strategy: str):
    """Calculates retrieval metrics for one chunking strategy."""
    rows = []
    for q, q_vec in zip(eval_queries, query_vecs):
        found20 = retrieved_doc_ids(strategy, q_vec, TOP_K)
        row = {
            "strategy": strategy,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found20,
            "precision@5": precision_at_k(found20[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found20[:5], q["relevant_docs"]),
            "MRR": mrr(found20, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found20[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

metrics_by_query = pd.concat([evaluate_strategy(name) for name in chunk_sets], ignore_index=True)
summary = metrics_by_query.groupby("strategy")[["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]].mean().reset_index()
summary

In [ ]:
by_kind = metrics_by_query.groupby(["strategy", "kind"])[["precision@5", "recall@5", "MRR"]].mean().reset_index()
by_kind

## Step 3. Manual chunk review and strategy selection

In [ ]:
for name, chunks in chunk_sets.items():
    print("\n===", name, "===")
    for c in chunks[:3]:
        print(f"doc_id={c['doc_id']} chunk_id={c['chunk_id']} title={c['title']}")
        print(c["page_content"][:450].replace("\n", " "))
        print("---")

In [ ]:
best_strategy = summary.sort_values(["recall@5", "MRR", "precision@5"], ascending=False).iloc[0]["strategy"]
print("Selected chunking strategy:", best_strategy)
print(summary.to_string(index=False))

### Step 3 conclusion

After running this notebook on the target machine, save the printed `summary` and `by_kind` tables, compare strategies by `recall@5`, `MRR`, `precision@5`, and manual chunk review. The selected strategy is used in step 4 for baseline retrieval and diagnostics.